# 1. Exploratory Data Analysis (EDA)
## Vietnam Real Estate Dataset

## 1.1 Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

import os
os.makedirs('../figures', exist_ok=True)

print('Libraries imported successfully!')

## 1.2 Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('../vietnam_housing_dataset.csv')

# Display basic info
print(f'Dataset Shape: {df.shape}')
print(f'Number of Records: {len(df):,}')
print(f'Number of Features: {len(df.columns)}')

print('\nColumns:')
print(df.columns.tolist())

## 1.3 Data Types Verification

In [ ]:
# Check data types
print('=== DATA TYPES ===\n')
print(df.dtypes)

## 1.4 Missing Values Analysis

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage (%)': missing_pct
})
print('=== MISSING VALUES ===\n')
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Percentage (%)', ascending=False))

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(12, 6))
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]

colors = plt.cm.RdYlGn(missing_pct / 100)
bars = ax.barh(missing_pct.index, missing_pct.values, color=colors)

for bar, val in zip(bars, missing_pct.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{val:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Missing Percentage (%)', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig('../figures/01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n>>> Chart saved: ../figures/01_missing_values.png')

## 1.5 Descriptive Statistics

In [ ]:
# Descriptive statistics for numerical columns
print('=== DESCRIPTIVE STATISTICS (Numerical) ===\n')
print(df.describe().round(2))

In [ ]:
# Descriptive statistics for categorical columns
print('\n=== DESCRIPTIVE STATISTICS (Categorical) ===\n')
categorical_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state']
for col in categorical_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False))

## 1.6 Price Distribution Analysis

In [ ]:
# Price Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Price'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Price (Billions VND)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Price Distribution (Histogram)', fontsize=13, fontweight='bold')
axes[0].axvline(df['Price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Price"].mean():.2f}')
axes[0].axvline(df['Price'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df["Price"].median():.2f}')
axes[0].legend()

# KDE Plot
sns.kdeplot(df['Price'].dropna(), ax=axes[1], fill=True, color='steelblue', alpha=0.6)
axes[1].set_xlabel('Price (Billions VND)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Price Distribution (KDE)', fontsize=13, fontweight='bold')
axes[1].axvline(df['Price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Price"].mean():.2f}')
axes[1].axvline(df['Price'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df["Price"].median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/02_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n>>> Chart saved: ../figures/02_price_distribution.png')

### Conclusion: Price Distribution
- Price distribution is **right-skewed** (positively skewed)
- Mean > Median indicates most properties are below average price
- Majority of properties fall in the lower price range

## 1.7 Area vs Price Analysis

In [ ]:
# Scatter Plot: Area vs Price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(df['Area'], df['Price'], alpha=0.3, c='steelblue', s=20)
axes[0].set_xlabel('Area (m\u00b2)', fontsize=11)
axes[0].set_ylabel('Price (Billions VND)', fontsize=11)
axes[0].set_title('Area vs Price (Scatter)', fontsize=13, fontweight='bold')

# Add trend line
z = np.polyfit(df['Area'].dropna(), df['Price'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(df['Area'].min(), df['Area'].quantile(0.95), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend Line')
axes[0].legend()

# Hexbin plot for density
hb = axes[1].hexbin(df['Area'], df['Price'], gridsize=30, cmap='YlOrRd', mincnt=1)
axes[1].set_xlabel('Area (m\u00b2)', fontsize=11)
axes[1].set_ylabel('Price (Billions VND)', fontsize=11)
axes[1].set_title('Area vs Price (Density)', fontsize=13, fontweight='bold')
plt.colorbar(hb, ax=axes[1], label='Count')

plt.tight_layout()
plt.savefig('../figures/03_area_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate correlation
corr = df['Area'].corr(df['Price'])
print(f'\nCorrelation between Area and Price: {corr:.4f}')

### Conclusion: Area vs Price
- Correlation coefficient shows weak positive correlation
- There is a slight positive relationship between area and price
- Price variation is high at all area levels, suggesting other factors influence price

## 1.8 Floors vs Price Analysis

In [ ]:
# Boxplot: Floors vs Price
fig, ax = plt.subplots(figsize=(12, 6))

# Filter to reasonable floor range
df_floors = df[df['Floors'].notna() & (df['Floors'] <= 10)]

sns.boxplot(x='Floors', y='Price', data=df_floors, palette='Blues', ax=ax)
ax.set_xlabel('Number of Floors', fontsize=11)
ax.set_ylabel('Price (Billions VND)', fontsize=11)
ax.set_title('Price Distribution by Number of Floors', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/04_floors_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistics by floors
print('\n=== Price Statistics by Floors ===\n')
print(df_floors.groupby('Floors')['Price'].agg(['count', 'mean', 'median', 'std']).round(2))

### Conclusion: Floors vs Price
- Most properties have 2-5 floors
- Price variation increases with number of floors
- Outliers present across all floor categories

## 1.9 Legal Status vs Price Analysis

In [ ]:
# Boxplot: Legal Status vs Price
fig, ax = plt.subplots(figsize=(10, 6))

df_legal = df[df['Legal status'].notna()]

order = df_legal.groupby('Legal status')['Price'].median().sort_values(ascending=False).index

sns.boxplot(x='Legal status', y='Price', data=df_legal, order=order, palette='Set2', ax=ax)
ax.set_xlabel('Legal Status', fontsize=11)
ax.set_ylabel('Price (Billions VND)', fontsize=11)
ax.set_title('Price Distribution by Legal Status', fontsize=13, fontweight='bold')
plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig('../figures/05_legal_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistics by legal status
print('\n=== Price Statistics by Legal Status ===\n')
print(df_legal.groupby('Legal status')['Price'].agg(['count', 'mean', 'median', 'std']).round(2).sort_values('mean', ascending=False))

### Conclusion: Legal Status vs Price
- Legal status significantly affects property value
- Properties with certificates typically have higher market value

## 1.10 Correlation Heatmap

In [ ]:
# Correlation Heatmap for numerical columns
numerical_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']
df_numerical = df[numerical_cols]

# Calculate correlation matrix
corr_matrix = df_numerical.corr()

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlGn', center=0,
            fmt='.3f', square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Correlation with Price ===\n')
price_corr = corr_matrix['Price'].sort_values(ascending=False)
print(price_corr)

### Conclusion: Correlation Analysis
- **Bedrooms** and **Bathrooms** show moderate positive correlation with Price
- **Area** has weak positive correlation with Price
- **Frontage** and **Access Road** have minimal correlation
- No strong multicollinearity detected between features

## 1.11 EDA Summary

In [ ]:
print('='*60)
print('EXPLORATORY DATA ANALYSIS - SUMMARY')
print('='*60)
print(f'\nTotal Records: {len(df):,}')
print(f'Total Features: {len(df.columns)}')
print(f'\nMissing Values: {df.isnull().sum().sum():,} cells')
print(f'\nPrice Range: {df["Price"].min():.2f} - {df["Price"].max():.2f} Billions VND')
print(f'Average Price: {df["Price"].mean():.2f} Billions VND')
print(f'Median Price: {df["Price"].median():.2f} Billions VND')
print(f'\nArea Range: {df["Area"].min():.1f} - {df["Area"].max():.1f} m\u00b2')
print(f'Average Area: {df["Area"].mean():.2f} m\u00b2')
print(f'\nKey Insights:')
print('  1. Price distribution is right-skewed')
print('  2. Area has weak positive correlation with Price')
print('  3. Legal status significantly affects price')
print('  4. Bedrooms and Bathrooms show moderate correlation with Price')
print('='*60)